<a href="https://colab.research.google.com/github/TanjaBioInfo/CodeCollection_BioinformaticsMaster/blob/main/SSO_Tarif_Versuch_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aus dem SSO-Tarif PDF die Zeilen/Tarife extrahieren

Jede Position ist eine Klasse

Beispiel für die erste Position (4.000):
from dataclasses import dataclass

@dataclass
class ZahnarztTarif:
    code: 4.000
    name: Befundaufnahme_Zweitmeinung
    punkte_min: 62.20
    punkte_max: 84.20
    wert_pro_punkt_min: 1.00
    wert_pro_punkt_max: 1.70

jede Behandlung ist eine Sammlung von einzelnen Tarifpositionen

## Load and Process PDF

### Subtask:
Load the '/content/tarif_SSO_Jan25.pdf' file and extract its content to prepare for data parsing.


**Reasoning**:
To load and extract content from the PDF file, I need to install the `PyPDF2` library first, as it's not a standard library and will be used to handle PDF operations.



In [141]:
import sys
!{sys.executable} -m pip install PyPDF2

print("PyPDF2 installed successfully.")

PyPDF2 installed successfully.


**Reasoning**:
Now that `PyPDF2` is installed, I will use it to open the PDF file, extract text from each page, and store it in a variable for further processing, as per the subtask instructions.



In [142]:
from PyPDF2 import PdfReader

pdf_path = '/content/tarif_SSO_Jan25.pdf'
pdf_text = ''

with open(pdf_path, 'rb') as file:
    reader = PdfReader(file)
    for page in reader.pages:
        pdf_text += page.extract_text()

print(f"First 2000 characters of the extracted PDF text:\n{pdf_text[:2000]}")

First 2000 characters of the extracted PDF text:
Titel: Suchmaschine für Zahnarzt-T arif UV/MV/IV
(Offline-T arifbrowser)
Tarifnummer: 222
Inhalt: Leistungskatalog Zahnarzt-T arif UV/MV/IV
Inkraftsetzung: 1. Januar 2018
Bearbeitungsstand: 18. Dezember 2024
Version: V2.00 / 1. Januar 2025
Tarifparteien: - Medizinaltarif-Kommission UVG (MTK)
- Suva, Abteilung Militärversicherung
- Bundesamt für Sozialversicherungen, Geschäftsfeld Invalidenversicherung
- Schweizerische Zahnärzte-Gesellschaft SSO
Realisiert durch die Zentralstelle für Medizinaltarife UVG (ZMT) / Kontakt: www .mtk-ctm.ch
Nutzungshinweis:
Der im vorliegenden Offline-T arifbrowser enthaltene Leistungskatalog des Zahnarzt-T arifs ist im Sozialversicherungsbereich UV/MV/IV anwendbar
und für die interessierte Öffentlichkeit frei einsehbar .
Gestützt darauf sind die Leistungserbringer berechtigt (d.h. SSO-Mitglieder und dem T arifvertrag angeschlossene Einzelkontrahenten), ihre bei UV/MV/IV -
Versicherten erbrachten Leistungen ge

## Aktualisiere die ZahnarztTarif-Definition

### Subtask:
Stelle sicher, dass die `ZahnarztTarif` Dataclass nur einmal korrekt definiert ist.


In [143]:
import re
from PyPDF2 import PdfReader
from dataclasses import dataclass

@dataclass
class ZahnarztTarif:
    code: str
    name: str
    punkte_min: float
    punkte_max: float
    wert_pro_punkt_min: float
    wert_pro_punkt_max: float

def parse_tarif_data(text):
    tarifs = []
    code_name_pattern = re.compile(r"^(\d{1,2}\.\d{3,5})\s*(.+)")
    punkte_min_pattern = re.compile(r"TP \(PP\) min (\d{1,3}(?:\.\d{1,2})?)")
    punkte_max_pattern = re.compile(r"TP \(PP\) max (\d{1,3}(?:\.\d{1,2})?)")

    current_tarif = None
    lines = text.splitlines()

    noise_patterns = [
        r"Tarif 222 - Zahnarzt-T arif UV/MV/IV \(SSO\)",
        r"Tarif 222 - Zahnarzt-Tarif UV/MV/IV \(SSO\)",
        r"Inhaltsverzeichnis",
        r"Zahnarzt-Tarif UV/MV/IV",
        r"Zahnarzt-T arif UV/MV/IV",
        r"MTK ZM T",
        r"www\.mtk-ctm\.ch",
        r"Seite \d+ von \d+",
        r"\d+\s+von\s+\d+",
        r"Gültigkeit \d{2}\.\d{2}\.\d{2}\s*-\s*\d{2}\.\d{2}\.\d{2}",
        r"TP \(UV/MV/IV\) \d{1,3}(?:\.\d{1,2})?",
        r"MwSt-Satz Kein Satz",
        r"Kapitel\s+\d{1,2}(?:\.\d{1,2})?.*?\d+\s*/\s*\d+",
        r"Kapitel\s+\d{1,2}(?:\.\d{1,2})?:",
        r"Deckung UV/MV/IV",
        r"Mitglied von LP",
        r"Referenz-Code",
        r"notwendig",
        r"Exklusive V erschluss",
        r"Gilt auch für Milchzähne",
        r"Wird von den V ersicherern nach UV/MV/IV nicht vergütet",
        r"Zahntechnische Laborleistung separat verrechenbar",
        r"Materialkosten mit Clusterposition \(Kap\. 20 \) separat verrechenbar",
        r"•[A-Za-zäöüÄÖÜß.,\s\(\)-]+",
        r"Leistung\s+\d{1,2}\.\d{3,5}\s+ist nicht kumulierbar mit"
    ]

    for i, raw_line in enumerate(lines):
        line = raw_line.strip()
        if not line:
            continue

        for noise in noise_patterns:
            line = re.sub(noise, '', line).strip()

        line = re.sub(r'\s+', ' ', line).strip()

        if not line or re.fullmatch(r'\d+', line):
            continue

        match_code_name = code_name_pattern.match(line)
        if match_code_name:
            if current_tarif and \
               current_tarif['punkte_min'] is not None and \
               current_tarif['punkte_max'] is not None:
                tarifs.append(current_tarif)

            name_raw = match_code_name.group(2).strip()
            name_clean = name_raw.replace('«', '').replace('»', '').strip().replace('  ', ' ')

            current_tarif = {
                'code': match_code_name.group(1),
                'name': name_clean,
                'punkte_min': None,
                'punkte_max': None,
                'wert_pro_punkt_min': 1.00,
                'wert_pro_punkt_max': 1.70
            }
            continue

        if current_tarif:
            match_punkte_min = punkte_min_pattern.match(line)
            if match_punkte_min:
                current_tarif['punkte_min'] = float(match_punkte_min.group(1))
                if current_tarif['punkte_max'] is not None:
                    tarifs.append(current_tarif)
                    current_tarif = None
                continue

            match_punkte_max = punkte_max_pattern.match(line)
            if match_punkte_max:
                current_tarif['punkte_max'] = float(match_punkte_max.group(1))
                if current_tarif['punkte_min'] is not None:
                    tarifs.append(current_tarif)
                    current_tarif = None
                continue

    if current_tarif and \
       current_tarif['punkte_min'] is not None and \
       current_tarif['punkte_max'] is not None:
        tarifs.append(current_tarif)
    return tarifs

# Load the PDF file and extract its content
pdf_path = '/content/tarif_SSO_Jan25.pdf'
pdf_text = ''

with open(pdf_path, 'rb') as file:
    reader = PdfReader(file)
    for page in reader.pages:
        pdf_text += page.extract_text()

# Parse the extracted PDF text to get tarif dictionaries
extracted_tarifs = parse_tarif_data(pdf_text)

# Convert extracted dictionaries to ZahnarztTarif instances
tarif_instances = []
for tarif_data in extracted_tarifs:
    tarif_instances.append(ZahnarztTarif(**tarif_data))

print(f"Extracted {len(tarif_instances)} tarif entries and created instances.")
if tarif_instances:
    print("First 5 ZahnarztTarif instances:")
    for instance in tarif_instances[:5]:
        print(instance)


Extracted 504 tarif entries and created instances.
First 5 ZahnarztTarif instances:
ZahnarztTarif(code='4.0000', name='Befundaufnahme; Zweitmeinung', punkte_min=62.2, punkte_max=84.2, wert_pro_punkt_min=1.0, wert_pro_punkt_max=1.7)
ZahnarztTarif(code='4.0010', name='Befundaufnahme beim Recallpatienten', punkte_min=41.5, punkte_max=56.1, wert_pro_punkt_min=1.0, wert_pro_punkt_max=1.7)
ZahnarztTarif(code='4.0020', name='Kurzbefundaufnahme', punkte_min=28.1, punkte_max=38.1, wert_pro_punkt_min=1.0, wert_pro_punkt_max=1.7)
ZahnarztTarif(code='4.0030', name='Kurzbefundaufnahme durch Zahnarzt anlässlich der', punkte_min=29.7, punkte_max=40.1, wert_pro_punkt_min=1.0, wert_pro_punkt_max=1.7)
ZahnarztTarif(code='4.0040', name='Befundaufnahme beim Notfallpatienten zwischen 20.00 Uhr und 07.00', punkte_min=124.4, punkte_max=168.4, wert_pro_punkt_min=1.0, wert_pro_punkt_max=1.7)
